<a href="https://colab.research.google.com/github/AKi-Dev-07/AKi-Dev-07/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpinRank Dataset Analysis
## 20 Problem Statements using Numpy and Pandas

This notebook analyzes the OpinRank dataset which contains car and hotel reviews. We will formulate 20 problem statements and solve them using Numpy and Pandas methods.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import os
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Step 1: Load and Explore the Dataset

In [ ]:
# Define base paths
base_path = r"c:\Users\adity\Downloads\opinrank+review+dataset\OpinRankDataset"
cars_path = os.path.join(base_path, "cars")
hotels_path = os.path.join(base_path, "hotels")

# List available years in cars
car_years = os.listdir(cars_path)
print("Available car years:", car_years)

# List available cities in hotels
hotel_cities = os.listdir(hotels_path)
print("Available hotel cities:", hotel_cities)

Available car years: ['2007', '2008', '2009']
Available hotel cities: ['beijing', 'chicago', 'dubai', 'las-vegas', 'london', 'montreal', 'new-delhi', 'new-york-city', 'san-francisco', 'shanghai']


In [ ]:
# Function to parse review files (both car and hotel formats)
def parse_review_file(file_path):
    """Parse a review file and extract reviews"""
    reviews = []
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()

            # Check if it's car format (XML-like with DOC tags)
            if '<DOC>' in content:
                # Parse car/hotel XML format
                doc_pattern = r'<DOC>(.*?)</DOC>'
                docs = re.findall(doc_pattern, content, re.DOTALL)
                for doc in docs:
                    date_match = re.search(r'<DATE>(.*?)</DATE>', doc)
                    author_match = re.search(r'<AUTHOR>(.*?)</AUTHOR>', doc)
                    text_match = re.search(r'<TEXT>(.*?)</TEXT>', doc)
                    fav_match = re.search(r'<FAVORITE>(.*?)</FAVORITE>', doc)

                    if text_match:
                        review_text = text_match.group(1).strip()
                        title = author_match.group(1).strip() if author_match else "Unknown"
                        date_str = date_match.group(1).strip() if date_match else ""

                        reviews.append({
                            'date': date_str,
                            'title': title,
                            'review': review_text
                        })
            else:
                # Parse hotel text format (tab-separated)
                parts = re.split(r'([A-Za-z]+ \d+ \d{4})', content)
                for i in range(1, len(parts), 3):
                    if i+1 < len(parts):
                        date_str = parts[i].strip()
                        rest = parts[i+1].strip()
                        if '\t' in rest:
                            idx = rest.index('\t')
                            title = rest[:idx].strip()
                            review = rest[idx+1:].strip()
                            reviews.append({
                                'date': date_str,
                                'title': title,
                                'review': review
                            })
    except Exception as e:
        pass
    return reviews

In [ ]:
# Load car reviews
car_reviews = []
for year in car_years:
    year_path = os.path.join(cars_path, year)
    if os.path.isdir(year_path):
        for file in os.listdir(year_path):
            file_path = os.path.join(year_path, file)
            if os.path.isfile(file_path):
                # Extract car name from filename (remove year prefix)
                car_name = file
                reviews = parse_review_file(file_path)
                for review in reviews:
                    review['category'] = 'car'
                    review['year'] = year
                    review['item_name'] = car_name
                    car_reviews.append(review)

print(f"Total car reviews loaded: {len(car_reviews)}")

Total car reviews loaded: 41727


In [ ]:
# Load hotel reviews
hotel_reviews = []
for city in hotel_cities:
    city_path = os.path.join(hotels_path, city)
    if os.path.isdir(city_path):
        for file in os.listdir(city_path):
            file_path = os.path.join(city_path, file)
            if os.path.isfile(file_path):
                # Extract hotel name from filename
                hotel_name = file
                reviews = parse_review_file(file_path)
                for review in reviews:
                    review['category'] = 'hotel'
                    review['city'] = city
                    review['item_name'] = hotel_name
                    hotel_reviews.append(review)

print(f"Total hotel reviews loaded: {len(hotel_reviews)}")

Total hotel reviews loaded: 76563


In [ ]:
# Create DataFrames
car_df = pd.DataFrame(car_reviews)
hotel_df = pd.DataFrame(hotel_reviews)

print("Car DataFrame shape:", car_df.shape)
print("Hotel DataFrame shape:", hotel_df.shape)

# Display sample data
print("\n--- Car DataFrame Sample ---")
print(car_df.head(3))
print("\n--- Hotel DataFrame Sample ---")
print(hotel_df.head(3))

Car DataFrame shape: (41727, 6)
Hotel DataFrame shape: (76563, 6)

--- Car DataFrame Sample ---
         date      title                                             review  \
0  07/31/2009    FlewByU  I just moved to Germany two months ago and bou...   
1  07/30/2009  cvillemdx  After months of careful research and test driv...   
2  06/22/2009    Pleased  I'm two years into a three year lease and I lo...   

  category  year       item_name  
0      car  2007  2007_acura_mdx  
1      car  2007  2007_acura_mdx  
2      car  2007  2007_acura_mdx  

--- Hotel DataFrame Sample ---
          date                                              title  \
0  Oct 12 2009            Nice trendy hotel location not too bad.   
1  Jul 17 2009       Stylish clean reasonable value poor location   
2  Nov 17 2009  great room layout service value-would definite...   

                                              review category     city  \
0  I stayed in this hotel for one night. As this ...    hotel  b

## Step 2: Data Preprocessing

In [ ]:
# Convert date columns to datetime
car_df['date'] = pd.to_datetime(car_df['date'], errors='coerce')
hotel_df['date'] = pd.to_datetime(hotel_df['date'], errors='coerce')

# Add review length feature
car_df['review_length'] = car_df['review'].apply(lambda x: len(str(x)) if pd.notna(x) else 0)
hotel_df['review_length'] = hotel_df['review'].apply(lambda x: len(str(x)) if pd.notna(x) else 0)

# Add title length feature
car_df['title_length'] = car_df['title'].apply(lambda x: len(str(x)) if pd.notna(x) else 0)
hotel_df['title_length'] = hotel_df['title'].apply(lambda x: len(str(x)) if pd.notna(x) else 0)

# Add word count feature
car_df['word_count'] = car_df['review'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)
hotel_df['word_count'] = hotel_df['review'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)

print("Car DataFrame after preprocessing:")
print(car_df.info())
print("\nHotel DataFrame after preprocessing:")
print(hotel_df.info())

Car DataFrame after preprocessing:
<class 'pandas.DataFrame'>
RangeIndex: 41727 entries, 0 to 41726
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           41727 non-null  datetime64[us]
 1   title          41727 non-null  str           
 2   review         41727 non-null  str           
 3   category       41727 non-null  str           
 4   year           41727 non-null  str           
 5   item_name      41727 non-null  str           
 6   review_length  41727 non-null  int64         
 7   title_length   41727 non-null  int64         
 8   word_count     41727 non-null  int64         
dtypes: datetime64[us](1), int64(3), str(5)
memory usage: 2.9 MB
None

Hotel DataFrame after preprocessing:
<class 'pandas.DataFrame'>
RangeIndex: 76563 entries, 0 to 76562
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         


## Step 3: Formulate and Solve 20 Problem Statements

### Problem 1: Find the average review length for cars vs hotels

In [ ]:
# Problem 1: Average review length comparison
print("=" * 60)
print("PROBLEM 1: Average Review Length for Cars vs Hotels")
print("=" * 60)

# Using Pandas
car_avg_length = car_df['review_length'].mean()
hotel_avg_length = hotel_df['review_length'].mean()

print(f"\nUsing Pandas:")
print(f"  Average car review length: {car_avg_length:.2f} characters")
print(f"  Average hotel review length: {hotel_avg_length:.2f} characters")

# Using Numpy
car_lengths_np = car_df['review_length'].values
hotel_lengths_np = hotel_df['review_length'].values

car_avg_np = np.mean(car_lengths_np)
hotel_avg_np = np.mean(hotel_lengths_np)

print(f"\nUsing Numpy:")
print(f"  Average car review length: {car_avg_np:.2f} characters")
print(f"  Average hotel review length: {hotel_avg_np:.2f} characters")

# Additional statistics using Numpy
print(f"\nAdditional Statistics (Numpy):")
print(f"  Car reviews - Min: {np.min(car_lengths_np)}, Max: {np.max(car_lengths_np)}, Std: {np.std(car_lengths_np):.2f}")
print(f"  Hotel reviews - Min: {np.min(hotel_lengths_np)}, Max: {np.max(hotel_lengths_np)}, Std: {np.std(hotel_lengths_np):.2f}")

PROBLEM 1: Average Review Length for Cars vs Hotels

Using Pandas:
  Average car review length: 466.25 characters
  Average hotel review length: 996.06 characters

Using Numpy:
  Average car review length: 466.25 characters
  Average hotel review length: 996.06 characters

Additional Statistics (Numpy):
  Car reviews - Min: 4, Max: 973, Std: 184.75
  Hotel reviews - Min: 2, Max: 18443, Std: 792.18


### Problem 2: Find the distribution of reviews by year for cars

In [ ]:
# Problem 2: Distribution of reviews by year for cars
print("=" * 60)
print("PROBLEM 2: Distribution of Car Reviews by Year")
print("=" * 60)

# Extract year from date
car_df['review_year'] = car_df['date'].dt.year

# Using Pandas value_counts
year_distribution = car_df['review_year'].value_counts().sort_index()
print(f"\nUsing Pandas value_counts:")
print(year_distribution)

# Using Numpy
years_np = car_df['review_year'].dropna().values
unique_years, counts = np.unique(years_np, return_counts=True)
print(f"\nUsing Numpy unique:")
for year, count in zip(unique_years, counts):
    print(f"  Year {int(year)}: {count} reviews")

# Calculate percentage distribution
total_reviews = len(car_df)
percentages = (counts / total_reviews) * 100
print(f"\nPercentage Distribution (Numpy):")
for year, pct in zip(unique_years, percentages):
    print(f"  Year {int(year)}: {pct:.2f}%")

PROBLEM 2: Distribution of Car Reviews by Year

Using Pandas value_counts:
review_year
2006     4359
2007    11615
2008    15465
2009    10288
Name: count, dtype: int64

Using Numpy unique:
  Year 2006: 4359 reviews
  Year 2007: 11615 reviews
  Year 2008: 15465 reviews
  Year 2009: 10288 reviews

Percentage Distribution (Numpy):
  Year 2006: 10.45%
  Year 2007: 27.84%
  Year 2008: 37.06%
  Year 2009: 24.66%


### Problem 3: Find the top 10 car models with the most reviews

In [ ]:
# Problem 3: Top 10 car models with most reviews
print("=" * 60)
print("PROBLEM 3: Top 10 Car Models with Most Reviews")
print("=" * 60)

# Using Pandas
top_cars = car_df['item_name'].value_counts().head(10)
print(f"\nUsing Pandas value_counts:")
print(top_cars)

# Using Numpy
car_names_np = car_df['item_name'].values
unique_cars, car_counts = np.unique(car_names_np, return_counts=True)
sorted_indices = np.argsort(car_counts)[::-1][:10]
print(f"\nUsing Numpy (Top 10):")
for idx in sorted_indices:
    print(f"  {unique_cars[idx]}: {car_counts[idx]} reviews")

PROBLEM 3: Top 10 Car Models with Most Reviews

Using Pandas value_counts:
item_name
2008_honda_civic          437
2007_honda_civic          432
2007_toyota_yaris         362
2007_honda_fit            355
2007_dodge_caliber        310
2007_honda_cr-v           299
2007_hyundai_santa_fe     295
2007_honda_accord         277
2007_toyota_fj_cruiser    275
2007_bmw_3_series         265
Name: count, dtype: int64

Using Numpy (Top 10):
  2008_honda_civic: 437 reviews
  2007_honda_civic: 432 reviews
  2007_toyota_yaris: 362 reviews
  2007_honda_fit: 355 reviews
  2007_dodge_caliber: 310 reviews
  2007_honda_cr-v: 299 reviews
  2007_hyundai_santa_fe: 295 reviews
  2007_honda_accord: 277 reviews
  2007_toyota_fj_cruiser: 275 reviews
  2007_bmw_3_series: 265 reviews


### Problem 4: Find the top 10 hotel cities with the most reviews

In [ ]:
# Problem 4: Top 10 hotel cities with most reviews
print("=" * 60)
print("PROBLEM 4: Top 10 Hotel Cities with Most Reviews")
print("=" * 60)

# Using Pandas
top_cities = hotel_df['city'].value_counts().head(10)
print(f"\nUsing Pandas value_counts:")
print(top_cities)

# Using Numpy
city_names_np = hotel_df['city'].values
unique_cities, city_counts = np.unique(city_names_np, return_counts=True)
sorted_indices = np.argsort(city_counts)[::-1][:10]
print(f"\nUsing Numpy (Top 10):")
for idx in sorted_indices:
    print(f"  {unique_cities[idx]}: {city_counts[idx]} reviews")

PROBLEM 4: Top 10 Hotel Cities with Most Reviews

Using Pandas value_counts:
city
london           22912
new-york-city    16853
san-francisco     9295
las-vegas         8465
chicago           5747
montreal          5557
dubai             3568
beijing           1535
new-delhi         1472
shanghai          1159
Name: count, dtype: int64

Using Numpy (Top 10):
  london: 22912 reviews
  new-york-city: 16853 reviews
  san-francisco: 9295 reviews
  las-vegas: 8465 reviews
  chicago: 5747 reviews
  montreal: 5557 reviews
  dubai: 3568 reviews
  beijing: 1535 reviews
  new-delhi: 1472 reviews
  shanghai: 1159 reviews


### Problem 5: Find the median review length for each car year

In [ ]:
# Problem 5: Median review length for each car year
print("=" * 60)
print("PROBLEM 5: Median Review Length for Each Car Year")
print("=" * 60)

# Using Pandas
median_by_year = car_df.groupby('review_year')['review_length'].median()
print(f"\nUsing Pandas groupby median:")
print(median_by_year)

# Using Numpy
print(f"\nUsing Numpy:")
for year in sorted(car_df['review_year'].dropna().unique()):
    year_data = car_df[car_df['review_year'] == year]['review_length'].values
    median_val = np.median(year_data)
    print(f"  Year {int(year)}: {median_val:.2f} characters")

PROBLEM 5: Median Review Length for Each Car Year

Using Pandas groupby median:
review_year
2006    387.0
2007    445.0
2008    500.0
2009    518.0
Name: review_length, dtype: float64

Using Numpy:
  Year 2006: 387.00 characters
  Year 2007: 445.00 characters
  Year 2008: 500.00 characters
  Year 2009: 518.00 characters


### Problem 6: Find the standard deviation of review lengths by city

In [ ]:
# Problem 6: Standard deviation of review lengths by city
print("=" * 60)
print("PROBLEM 6: Standard Deviation of Review Lengths by City")
print("=" * 60)

# Using Pandas
std_by_city = hotel_df.groupby('city')['review_length'].std()
print(f"\nUsing Pandas groupby std:")
print(std_by_city.sort_values(ascending=False))

# Using Numpy
print(f"\nUsing Numpy:")
for city in hotel_df['city'].unique():
    city_data = hotel_df[hotel_df['city'] == city]['review_length'].values
    std_val = np.std(city_data)
    print(f"  {city}: {std_val:.2f}")

PROBLEM 6: Standard Deviation of Review Lengths by City

Using Pandas groupby std:
city
dubai            1029.552080
shanghai         1020.554169
las-vegas         933.951453
beijing           864.548358
chicago           790.994005
new-delhi         750.094689
montreal          747.871816
new-york-city     743.159669
london            738.683619
san-francisco     718.794054
Name: review_length, dtype: float64

Using Numpy:
  beijing: 864.27
  chicago: 790.93
  dubai: 1029.41
  las-vegas: 933.90
  london: 738.67
  montreal: 747.80
  new-delhi: 749.84
  new-york-city: 743.14
  san-francisco: 718.76
  shanghai: 1020.11


### Problem 7: Find the month with the most car reviews

In [ ]:
# Problem 7: Month with the most car reviews
print("=" * 60)
print("PROBLEM 7: Month with Most Car Reviews")
print("=" * 60)

# Extract month from date
car_df['review_month'] = car_df['date'].dt.month

# Using Pandas
month_distribution = car_df['review_month'].value_counts().sort_index()
print(f"\nUsing Pandas value_counts:")
print(month_distribution)

# Find the month with most reviews
max_month = month_distribution.idxmax()
max_count = month_distribution.max()
month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
               7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}
print(f"\nMonth with most reviews: {month_names[max_month]} ({max_count} reviews)")

# Using Numpy
months_np = car_df['review_month'].dropna().values
unique_months, month_counts = np.unique(months_np, return_counts=True)
max_month_np = unique_months[np.argmax(month_counts)]
print(f"\nUsing Numpy: Month {int(max_month_np)} has the most reviews ({np.max(month_counts)})")

### Problem 8: Find the average word count for hotel reviews by city

In [ ]:
# Problem 8: Average word count for hotel reviews by city
print("=" * 60)
print("PROBLEM 8: Average Word Count for Hotel Reviews by City")
print("=" * 60)

# Using Pandas
avg_word_by_city = hotel_df.groupby('city')['word_count'].mean()
print(f"\nUsing Pandas groupby mean:")
print(avg_word_by_city.sort_values(ascending=False))

# Using Numpy
print(f"\nUsing Numpy:")
for city in hotel_df['city'].unique():
    city_data = hotel_df[hotel_df['city'] == city]['word_count'].values
    avg_val = np.mean(city_data)
    print(f"  {city}: {avg_val:.2f} words")

### Problem 9: Find the car model with the longest average review

In [ ]:
# Problem 9: Car model with longest average review
print("=" * 60)
print("PROBLEM 9: Car Model with Longest Average Review")
print("=" * 60)

# Using Pandas
avg_length_by_car = car_df.groupby('item_name')['review_length'].mean()
top_car = avg_length_by_car.idxmax()
top_length = avg_length_by_car.max()
print(f"\nUsing Pandas groupby mean:")
print(f"  Car with longest average review: {top_car}")
print(f"  Average review length: {top_length:.2f} characters")

# Using Numpy
car_names = car_df['item_name'].values
review_lengths = car_df['review_length'].values
unique_cars = np.unique(car_names)
avg_lengths = []
for car in unique_cars:
    mask = car_names == car
    avg_lengths.append(np.mean(review_lengths[mask]))
max_idx = np.argmax(avg_lengths)
print(f"\nUsing Numpy:")
print(f"  Car with longest average review: {unique_cars[max_idx]}")
print(f"  Average review length: {avg_lengths[max_idx]:.2f} characters")

### Problem 10: Find the distribution of review titles by length

In [ ]:
# Problem 10: Distribution of review titles by length
print("=" * 60)
print("PROBLEM 10: Distribution of Review Titles by Length")
print("=" * 60)

# Create length categories for titles
car_df['title_length_cat'] = pd.cut(car_df['title_length'],
                                      bins=[0, 10, 20, 30, 50, 100],
                                      labels=['Very Short (0-10)', 'Short (10-20)',
                                              'Medium (20-30)', 'Long (30-50)', 'Very Long (50+)'])

# Using Pandas
title_dist = car_df['title_length_cat'].value_counts()
print(f"\nUsing Pandas cut and value_counts:")
print(title_dist)

# Using Numpy
title_lengths = car_df['title_length'].values
bins = [0, 10, 20, 30, 50, 100]
hist, _ = np.histogram(title_lengths, bins=bins)
labels = ['Very Short (0-10)', 'Short (10-20)', 'Medium (20-30)', 'Long (30-50)', 'Very Long (50+)']
print(f"\nUsing Numpy histogram:")
for label, count in zip(labels, hist):
    print(f"  {label}: {count}")

### Problem 11: Find the correlation between review length and title length

In [ ]:
# Problem 11: Correlation between review length and title length
print("=" * 60)
print("PROBLEM 11: Correlation between Review Length and Title Length")
print("=" * 60)

# Using Pandas
car_corr = car_df['review_length'].corr(car_df['title_length'])
hotel_corr = hotel_df['review_length'].corr(hotel_df['title_length'])
print(f"\nUsing Pandas corr:")
print(f"  Car reviews correlation: {car_corr:.4f}")
print(f"  Hotel reviews correlation: {hotel_corr:.4f}")

# Using Numpy
car_review_len = car_df['review_length'].values
car_title_len = car_df['title_length'].values
hotel_review_len = hotel_df['review_length'].values
hotel_title_len = hotel_df['title_length'].values

# Calculate correlation using Numpy
car_corr_np = np.corrcoef(car_review_len, car_title_len)[0, 1]
hotel_corr_np = np.corrcoef(hotel_review_len, hotel_title_len)[0, 1]
print(f"\nUsing Numpy corrcoef:")
print(f"  Car reviews correlation: {car_corr_np:.4f}")
print(f"  Hotel reviews correlation: {hotel_corr_np:.4f}")

### Problem 12: Find the 25th, 50th, and 75th percentiles of review lengths

In [ ]:
# Problem 12: Percentiles of review lengths
print("=" * 60)
print("PROBLEM 12: Percentiles of Review Lengths")
print("=" * 60)

# Using Pandas
car_percentiles = car_df['review_length'].quantile([0.25, 0.5, 0.75])
hotel_percentiles = hotel_df['review_length'].quantile([0.25, 0.5, 0.75])
print(f"\nUsing Pandas quantile:")
print(f"  Car review length percentiles:")
print(f"    25th: {car_percentiles[0.25]:.2f}")
print(f"    50th (Median): {car_percentiles[0.5]:.2f}")
print(f"    75th: {car_percentiles[0.75]:.2f}")
print(f"  Hotel review length percentiles:")
print(f"    25th: {hotel_percentiles[0.25]:.2f}")
print(f"    50th (Median): {hotel_percentiles[0.5]:.2f}")
print(f"    75th: {hotel_percentiles[0.75]:.2f}")

# Using Numpy
car_lengths = car_df['review_length'].values
hotel_lengths = hotel_df['review_length'].values
car_percentiles_np = np.percentile(car_lengths, [25, 50, 75])
hotel_percentiles_np = np.percentile(hotel_lengths, [25, 50, 75])
print(f"\nUsing Numpy percentile:")
print(f"  Car review length percentiles: 25th={car_percentiles_np[0]:.2f}, 50th={car_percentiles_np[1]:.2f}, 75th={car_percentiles_np[2]:.2f}")
print(f"  Hotel review length percentiles: 25th={hotel_percentiles_np[0]:.2f}, 50th={hotel_percentiles_np[1]:.2f}, 75th={hotel_percentiles_np[2]:.2f}")

### Problem 13: Find the car year with the highest average review length

In [ ]:
# Problem 13: Car year with highest average review length
print("=" * 60)
print("PROBLEM 13: Car Year with Highest Average Review Length")
print("=" * 60)

# Using Pandas
avg_by_year = car_df.groupby('year')['review_length'].mean()
max_year = avg_by_year.idxmax()
max_avg = avg_by_year.max()
print(f"\nUsing Pandas groupby:")
print(f"  Year with highest average review length: {max_year}")
print(f"  Average review length: {max_avg:.2f} characters")

# Using Numpy
years = car_df['year'].values
lengths = car_df['review_length'].values
unique_years = np.unique(years)
year_avgs = []
for year in unique_years:
    mask = years == year
    year_avgs.append(np.mean(lengths[mask]))
max_idx = np.argmax(year_avgs)
print(f"\nUsing Numpy:")
print(f"  Year with highest average review length: {unique_years[max_idx]}")
print(f"  Average review length: {year_avgs[max_idx]:.2f} characters")

### Problem 14: Find the variance in word counts for car reviews

In [ ]:
# Problem 14: Variance in word counts for car reviews
print("=" * 60)
print("PROBLEM 14: Variance in Word Counts for Car Reviews")
print("=" * 60)

# Using Pandas
car_word_variance = car_df['word_count'].var()
hotel_word_variance = hotel_df['word_count'].var()
print(f"\nUsing Pandas var:")
print(f"  Car reviews word count variance: {car_word_variance:.2f}")
print(f"  Hotel reviews word count variance: {hotel_word_variance:.2f}")

# Using Numpy
car_words = car_df['word_count'].values
hotel_words = hotel_df['word_count'].values
car_var_np = np.var(car_words)
hotel_var_np = np.var(hotel_words)
print(f"\nUsing Numpy var:")
print(f"  Car reviews word count variance: {car_var_np:.2f}")
print(f"  Hotel reviews word count variance: {hotel_var_np:.2f}")

### Problem 15: Find the most common first words in review titles

In [ ]:
# Problem 15: Most common first words in review titles
print("=" * 60)
print("PROBLEM 15: Most Common First Words in Review Titles")
print("=" * 60)

# Extract first words from titles
car_df['first_word'] = car_df['title'].apply(lambda x: str(x).split()[0].lower() if pd.notna(x) and len(str(x).split()) > 0 else '')
hotel_df['first_word'] = hotel_df['title'].apply(lambda x: str(x).split()[0].lower() if pd.notna(x) and len(str(x).split()) > 0 else '')

# Using Pandas
car_first_words = car_df['first_word'].value_counts().head(10)
hotel_first_words = hotel_df['first_word'].value_counts().head(10)
print(f"\nUsing Pandas value_counts:")
print(f"  Car reviews - Top 10 first words:")
print(car_first_words)
print(f"\n  Hotel reviews - Top 10 first words:")
print(hotel_first_words)

# Using Numpy
car_first_np = car_df['first_word'].values
unique_words, word_counts = np.unique(car_first_np, return_counts=True)
sorted_indices = np.argsort(word_counts)[::-1][:10]
print(f"\nUsing Numpy (Car reviews - Top 10):")
for idx in sorted_indices:
    print(f"  {unique_words[idx]}: {word_counts[idx]}")

### Problem 16: Find the day of week with the most reviews

In [ ]:
# Problem 16: Day of week with most reviews
print("=" * 60)
print("PROBLEM 16: Day of Week with Most Reviews")
print("=" * 60)

# Extract day of week
car_df['day_of_week'] = car_df['date'].dt.day_name()
hotel_df['day_of_week'] = hotel_df['date'].dt.day_name()

# Using Pandas
car_day_dist = car_df['day_of_week'].value_counts()
hotel_day_dist = hotel_df['day_of_week'].value_counts()
print(f"\nUsing Pandas value_counts:")
print(f"  Car reviews by day of week:")
print(car_day_dist)
print(f"\n  Hotel reviews by day of week:")
print(hotel_day_dist)

# Find max day
car_max_day = car_day_dist.idxmax()
hotel_max_day = hotel_day_dist.idxmax()
print(f"\n  Car reviews most common on: {car_max_day}")
print(f"  Hotel reviews most common on: {hotel_max_day}")

# Using Numpy
car_days = car_df['day_of_week'].values
unique_days, day_counts = np.unique(car_days, return_counts=True)
max_day_idx = np.argmax(day_counts)
print(f"\nUsing Numpy: Most common day for car reviews is {unique_days[max_day_idx]}")

### Problem 17: Find the total number of reviews per category (car/hotel)

In [ ]:
# Problem 17: Total number of reviews per category
print("=" * 60)
print("PROBLEM 17: Total Number of Reviews per Category")
print("=" * 60)

# Combine dataframes
car_df['source'] = 'car'
hotel_df['source'] = 'hotel'
combined_df = pd.concat([car_df, hotel_df], ignore_index=True)

# Using Pandas
category_counts = combined_df['source'].value_counts()
print(f"\nUsing Pandas value_counts:")
print(category_counts)

# Using Numpy
sources = combined_df['source'].values
unique_sources, source_counts = np.unique(sources, return_counts=True)
print(f"\nUsing Numpy unique:")
for source, count in zip(unique_sources, source_counts):
    print(f"  {source}: {count} reviews")

# Calculate percentages
total = len(combined_df)
print(f"\nPercentage distribution:")
for source, count in zip(unique_sources, source_counts):
    pct = (count / total) * 100
    print(f"  {source}: {pct:.2f}%")

### Problem 18: Find the range of review lengths (max - min)

In [ ]:
# Problem 18: Range of review lengths
print("=" * 60)
print("PROBLEM 18: Range of Review Lengths")
print("=" * 60)

# Using Pandas
car_range = car_df['review_length'].max() - car_df['review_length'].min()
hotel_range = hotel_df['review_length'].max() - hotel_df['review_length'].min()
print(f"\nUsing Pandas (max - min):")
print(f"  Car review length range: {car_range} characters")
print(f"  Hotel review length range: {hotel_range} characters")

# Using Numpy
car_lengths = car_df['review_length'].values
hotel_lengths = hotel_df['review_length'].values
car_range_np = np.max(car_lengths) - np.min(car_lengths)
hotel_range_np = np.max(hotel_lengths) - np.min(hotel_lengths)
print(f"\nUsing Numpy (ptp - peak to peak):")
print(f"  Car review length range: {car_range_np} characters")
print(f"  Hotel review length range: {hotel_range_np} characters")

# Also using np.ptp
print(f"\nUsing np.ptp:")
print(f"  Car: {np.ptp(car_lengths)}")
print(f"  Hotel: {np.ptp(hotel_lengths)}")

### Problem 19: Find the skewness of review length distribution

In [ ]:
# Problem 19: Skewness of review length distribution
print("=" * 60)
print("PROBLEM 19: Skewness of Review Length Distribution")
print("=" * 60)

# Using Pandas
car_skew = car_df['review_length'].skew()
hotel_skew = hotel_df['review_length'].skew()
print(f"\nUsing Pandas skew:")
print(f"  Car review length skewness: {car_skew:.4f}")
print(f"  Hotel review length skewness: {hotel_skew:.4f}")

# Using Numpy (calculate skewness manually)
def numpy_skew(data):
    n = len(data)
    mean = np.mean(data)
    std = np.std(data)
    skew = np.mean(((data - mean) / std) ** 3)
    return skew

car_skew_np = numpy_skew(car_lengths)
hotel_skew_np = numpy_skew(hotel_lengths)
print(f"\nUsing Numpy (manual calculation):")
print(f"  Car review length skewness: {car_skew_np:.4f}")
print(f"  Hotel review length skewness: {hotel_skew_np:.4f}")

### Problem 20: Find the kurtosis of review length distribution

In [ ]:
# Problem 20: Kurtosis of review length distribution
print("=" * 60)
print("PROBLEM 20: Kurtosis of Review Length Distribution")
print("=" * 60)

# Using Pandas
car_kurt = car_df['review_length'].kurt()
hotel_kurt = hotel_df['review_length'].kurt()
print(f"\nUsing Pandas kurt:")
print(f"  Car review length kurtosis: {car_kurt:.4f}")
print(f"  Hotel review length kurtosis: {hotel_kurt:.4f}")

# Using Numpy (calculate kurtosis manually)
def numpy_kurtosis(data):
    n = len(data)
    mean = np.mean(data)
    std = np.std(data)
    kurt = np.mean(((data - mean) / std) ** 4) - 3
    return kurt

car_kurt_np = numpy_kurtosis(car_lengths)
hotel_kurt_np = numpy_kurtosis(hotel_lengths)
print(f"\nUsing Numpy (manual calculation):")
print(f"  Car review length kurtosis: {car_kurt_np:.4f}")
print(f"  Hotel review length kurtosis: {hotel_kurt_np:.4f}")

## Summary of Results

In [ ]:
# Summary of all problem statements and solutions
print("=" * 70)
print("SUMMARY: 20 Problem Statements and Solutions")
print("=" * 70)

summary_data = {
    'Problem #': list(range(1, 21)),
    'Problem Statement': [
        'Average review length for cars vs hotels',
        'Distribution of reviews by year for cars',
        'Top 10 car models with most reviews',
        'Top 10 hotel cities with most reviews',
        'Median review length for each car year',
        'Standard deviation of review lengths by city',
        'Month with most car reviews',
        'Average word count for hotel reviews by city',
        'Car model with longest average review',
        'Distribution of review titles by length',
        'Correlation between review length and title length',
        '25th, 50th, and 75th percentiles of review lengths',
        'Car year with highest average review length',
        'Variance in word counts for car reviews',
        'Most common first words in review titles',
        'Day of week with most reviews',
        'Total number of reviews per category',
        'Range of review lengths',
        'Skewness of review length distribution',
        'Kurtosis of review length distribution'
    ],
    'Pandas Method Used': [
        'mean()',
        'value_counts()',
        'value_counts().head(10)',
        'value_counts().head(10)',
        'groupby().median()',
        'groupby().std()',
        'value_counts().idxmax()',
        'groupby().mean()',
        'groupby().mean().idxmax()',
        'cut(), value_counts()',
        'corr()',
        'quantile()',
        'groupby().mean().idxmax()',
        'var()',
        'value_counts().head(10)',
        'value_counts().idxmax()',
        'value_counts()',
        'max() - min()',
        'skew()',
        'kurt()'
    ],
    'Numpy Method Used': [
        'np.mean()',
        'np.unique()',
        'np.unique(), np.argsort()',
        'np.unique(), np.argsort()',
        'np.median()',
        'np.std()',
        'np.unique(), np.argmax()',
        'np.mean()',
        'np.unique(), np.mean()',
        'np.histogram()',
        'np.corrcoef()',
        'np.percentile()',
        'np.unique(), np.mean()',
        'np.var()',
        'np.unique(), np.argsort()',
        'np.unique(), np.argmax()',
        'np.unique()',
        'np.ptp()',
        'Manual calculation',
        'Manual calculation'
    ]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))